# Simulation

## Random Variables {#sec-07-top}

In [ ]:
from math import pi 
import pprint

import mesa

import seaborn as sns
import matplotlib.pyplot as plt

import nltk
from nltk import word_tokenize

from numpy.random import default_rng
import numpy as np
import pandas as pd
from scipy import stats
from scipy.stats import binom, bernoulli, norm, expon, uniform

### Discrete Random Variables

In [ ]:
#| fig-align: center
#| fig-cap: Pmf for a biased coin. Note that there are only two possible values of X. This is the support of this random variable.
#| label: fig-pmf-biased-coin
#| fig-pos: 'ht'

plt.figure(figsize=(5,3))
plt.bar([0,1], [1/3, 2/3], tick_label=['0', '1'], width=0.3);
plt.xlim(-1,2); 
plt.ylim(0,1); 
plt.title('pmf of \'X\' random variable' );

In [ ]:
#| fig-align: center
#| fig-cap: Pmf for 10 tosses of a biased coin. Note that there are eleven possible values of Y.
#| label: fig-pmf-ten-tosses
#| fig-pos: 'ht'

plt.figure(figsize=(5,3))
probs = binom.pmf(np.arange(0, 11), n=10, p=2/3)
plt.bar(np.arange(0, 11), probs);plt.ylim(0,1);
plt.title('pmf of number of heads');

### Continuous Random Variables

In [ ]:
#| fig-align: center
#| fig-cap: "Continuous random variables"
#| label: fig-cts-rv
#| fig-pos: 'ht'

x = np.linspace(-3, 3, num=100)
y = norm.pdf(x)
plt.figure(figsize=(8,3))
plt.subplot(131)
plt.plot(x,y)
plt.ylim(0,1)
plt.title('Normal pdf')

y = expon.pdf(x[x>0])
plt.subplot(132)
plt.plot(x[x>0],y)
plt.ylim(0,1)
plt.title('Exponential pdf');

x = np.linspace(0, 1, num=100)
y = uniform.pdf(x)
plt.subplot(133)
plt.plot(x,y)
plt.ylim(0,1.1)
plt.title('Uniform pdf');

### Generating Random Variates

In [ ]:
#| fig-align: center
#| fig-cap: "Histogram of random variables"
#| label: fig-cts-hist
#| fig-pos: 'ht'

rng = default_rng(5003)

yvals = np.zeros((3,300))
yvals[0,:] = rng.normal(size=300)
yvals[1,:] = rng.exponential(size=300)
yvals[2, :] = rng.uniform(size=300)
pdfs = ['Normal', 'Exponential', 'Uniform']
plt.figure(figsize=(8,3))

for i in np.arange(0,3):
    plt.subplot(1, 3, i+1)
    plt.hist(yvals[i,:], density=True, histtype='step')
    plt.title(pdfs[i])
    plt.ylim(0, 1.05)

## General Principles in Simulation Studies
### Introduction
### Example: Insurance claims
### Example: Sandwich Shop Closing Time
### Steps in a Simulation Study
### Theory
#### Strong Law of Large Numbers
#### Central Limit Theorem
#### Sample Estimates
## Object-Oriented Programming in Python

In [ ]:
class Circle:
    """ A simple class definition 
    
    c0 = Circle()
    c0.radius
    
    """
    def __init__(self, radius = 1.0):
        self.radius = radius
        
    def area(self):
        """ Compute area of circle"""
        return pi*(self.radius**2)

In [ ]:
c1 = Circle(3.2)
c2 = Circle(4.0)

print(f"""
The radius of c1 is {c1.radius} and the radius of c2 is {c2.radius}.
The area of c1 is {c1.area():.3f} and the area of c2 is {c2.area():.3f}.
""")

## Introduction to Agent Based Models
### Introduction to Mesa
### Example: Boltzmann model overview
## Agent and Model Classes (v1)

In [ ]:
class MoneyAgent(mesa.Agent):
    """An agent with fixed initial wealth."""

    def __init__(self, model):
        # Pass the parameters to the parent class.
        super().__init__(model)

        # Create the agent's variable and set the initial values.
        self.wealth = 1

    def step(self):
        # Verify agent has some wealth
        if self.wealth > 0:
            other_agent = self.rng.choice(self.model.agents)
            other_agent.wealth += 1
            self.wealth -= 1

In [ ]:
class MoneyModel(mesa.Model):
    """A model with some number of agents."""

    def __init__(self, N, rng=None):
        super().__init__(rng=rng)
        self.num_agents = N

        # Create agents
        MoneyAgent.create_agents(model=self, n=N)

    def step(self):
        """Advance the model by one step."""
        self.agents.shuffle_do("step")

### Example: Boltzmann model simple execution

In [ ]:
rng2 = default_rng(5023)
all_wealth = []
model = MoneyModel(N=10, rng=rng2)

for i in range(100):
    model.step()

# Extract the results
for agent in model.agents:
    all_wealth.append(agent.wealth)
    
prop = (np.array(all_wealth) == 0).mean()
print(f"The proportion of agents with zero wealth is {prop:.3f}.")

### Data Collection

In [ ]:
def compute_zero_prop(mesa_model):
    all_wealth = []
    # Extract the results
    for agent in mesa_model.agents:
        all_wealth.append(agent.wealth)
    prop = (np.array(all_wealth) == 0).mean()
    return prop

compute_zero_prop(model)

In [ ]:
class MoneyModel(mesa.Model):
    """A model with some number of agents."""

    def __init__(self, N, rng=None):
        super().__init__(rng=rng)
        self.num_agents = N

        # Create agents
        MoneyAgent.create_agents(model=self, n=N)

        # initialise the data collector, telling it to use the 
        # compute_zero_prop() function
        # on the model.
        self.datacollector = mesa.DataCollector(
            model_reporters={"zero_prop": compute_zero_prop}
        )

    def step(self):
        self.agents.shuffle_do("step")
        # Collect data at every step
        self.datacollector.collect(self)

### Example: Boltzmann model data collection

In [ ]:
model = MoneyModel(N=10, rng=35)

for i in range(100):
    model.step()
    
df_output = model.datacollector.get_model_vars_dataframe()
df_output.head()

## Multiple Iterations
### Example: Boltzmann model multiple iterations

In [ ]:
params = {"N": 10}

results = mesa.batch_run(
    MoneyModel,
    parameters=params,
    rng= [None]*50,
    max_steps=100,
    number_processes=1,
    data_collection_period=-1
    #display_progress=True,
)
results_df = pd.DataFrame(results)

In [ ]:
#| fig-align: center
#| fig-cap: "Proportion of zero wealth agents, over 100 iterations"
#| label: fig-prop-zero-mult-iterations
#| fig-pos: 'ht'
results_df.zero_prop.hist(grid=False, 
                          bins=np.arange(0.10, 1.00, 0.05),
                          figsize=(4,3));

In [ ]:
ttest_out = stats.ttest_1samp(results_df.zero_prop, 0.0).confidence_interval()
print(f"""
The 95% CI for the mean proportion of agents with zero income is
({ttest_out[0]:.3f}, {ttest_out[1]:.3f})
       """)

## Agent and Model Classes (v2)
### Assessment of Income Inequality

In [ ]:
def compute_gini(model):
    agent_wealths = [agent.wealth for agent in model.agents]
    x = sorted(agent_wealths)
    N = model.num_agents
    B = sum(xi * (N - i) for i, xi in enumerate(x)) / (N * sum(x))
    return 1 + (1 / N) - 2 * B

class MoneyModel(mesa.Model):
    """A model with some number of agents."""

    def __init__(self, N, rng=None):
        super().__init__(rng=rng)
        self.num_agents = N

        # Create agents
        MoneyAgent.create_agents(model=self, n=N)

        # initialise the data collector, telling it to use both the gini() 
        # and compute_zero_prop() function
        # on the model.
        self.datacollector = mesa.DataCollector(
            model_reporters={"Gini": compute_gini, 
                             "zero_prop": compute_zero_prop}
        )

    def step(self):
        self.agents.shuffle_do("step")
        # Collect data at every step
        self.datacollector.collect(self)

### Data Collection
### Example: Boltzmann model data collection 

In [ ]:
params = {"N": 10}

results = mesa.batch_run(
    MoneyModel,
    parameters=params,
    rng = [None]*50,
    max_steps=100,
    number_processes=1,
    data_collection_period=1
    #display_progress=True,
)

In [ ]:
#| fig-align: center
#| fig-cap: "Evolution of Gini coefficient"
#| label: fig-gini-coef
#| fig-pos: 'ht'

results_df = pd.DataFrame(results)
grp_by_step = results_df.Gini.groupby(results_df.Step).describe()
#grp_by_step.head()

grp_by_step[['25%', '50%', '75%']].plot(style=['--', '-', '--'])
plt.ylim([0, 1]);

## Agent and Model Classes (v3)
### Adding a spatial component

In [ ]:
class MoneyModel(mesa.Model):
    """A model with some number of agents."""

    def __init__(self, N, width, height, rng=None):
        super().__init__(rng=rng)
        self.num_agents = N
        self.grid = mesa.space.MultiGrid(width, height, True)

        # Create agents
        for i in range(self.num_agents):
            a = MoneyAgent(self)
            # Add the agent to a random grid cell
            x = self.random.randrange(self.grid.width)
            y = self.random.randrange(self.grid.height)
            self.grid.place_agent(a, (x, y))
            
        # Data collectors for the model
        self.datacollector = mesa.DataCollector(
            model_reporters={"Gini": compute_gini}, 
            agent_reporters={"Wealth": "wealth"}
        )

    def step(self):
        self.datacollector.collect(self)
        self.agents.shuffle_do("step")

In [ ]:
class MoneyAgent(mesa.Agent):
    """An agent with fixed initial wealth."""

    def __init__(self, model):
        super().__init__(model)
        self.wealth = 1

    def move(self):
        possible_steps = self.model.grid.get_neighborhood(
            self.pos, moore=True, include_center=False
        )
        new_position = self.random.choice(possible_steps)
        self.model.grid.move_agent(self, new_position)

    def give_money(self):
        cellmates = self.model.grid.get_cell_list_contents([self.pos])
        cellmates.pop(
            cellmates.index(self)
        )  # Ensure agent is not giving money to itself
        if len(cellmates) >= 1:
            other = self.random.choice(cellmates)
            other.wealth += 1
            self.wealth -= 1
                
    # First, the agent moves. Then, it might give money away to another agent 
    # in a neighbouring cell.
    def step(self):
        self.move()
        if self.wealth > 0:
            self.give_money()

### Data Collection

In [ ]:
#| fig-align: center
#| fig-cap: "Evolution of Gini coefficient"
#| label: fig-gini-coef-2
#| fig-pos: 'ht'

model = MoneyModel(100, 10, 10)
for i in range(100):
    model.step()

gini = model.datacollector.get_model_vars_dataframe()

# Plot the Gini coefficient over time
g = sns.lineplot(data=gini)
g.set(title="Gini Coefficient over Time", ylabel="Gini Coefficient");

### Example: Boltzmann model data collection 

In [ ]:
#| eval: false
params = {"width": 10, "height": 10, "N": range(5, 20, 5)}

results = mesa.batch_run(
    MoneyModel,
    parameters=params,
    rng=[None]*50,
    max_steps=100,
    number_processes=1,
    data_collection_period=1#,
    #display_progress=True,
)

In [ ]:
#| fig-align: center
#| fig-cap: "Evolution of Gini over time steps"
#| label: fig-gini-over-time-steps

results_df = pd.DataFrame(results)
grp_by_step = results_df.Gini.groupby([results_df.Step, results_df.N]).describe()
grp_by_step.reset_index(inplace=True)

grouped = grp_by_step.groupby('N')
plt.figure(figsize=(10, 6))

for group in grouped:
    plt.plot(group[1]['Step'], group[1]['mean'], label=group[0], alpha=0.5)

plt.xlabel('Step')
plt.ylabel('Gini')
plt.title('Gini vs Step by N (number of agents)');
plt.legend();

### Example: Boltzmann model visualisation

In [ ]:
#| eval: false


from mesa.visualization import SolaraViz, SpaceRenderer, make_plot_component
from mesa.visualization.components import AgentPortrayalStyle

def agent_portrayal(agent):
    portrayal = AgentPortrayalStyle(size=50, color="tab:orange")
    if agent.wealth == 0:
        portrayal.update(("color", "tab:blue"), ("size", 100))
    return portrayal

model_params = {
    "N": {
        "type": "SliderInt",
        "value": 50,
        "label": "Number of agents:",
        "min": 10,
        "max": 100,
        "step": 1,
    },
    "width": 10,
    "height": 10,
}

money_model = MoneyModel(N=50, width=10, height=10)  # keyword arguments

renderer = (
    SpaceRenderer(model=money_model, backend="matplotlib")
    .setup_agents(agent_portrayal)
    .render()
)

GiniPlot = make_plot_component("Gini", page=0)

page = SolaraViz(
    money_model,
    renderer,
    components=[GiniPlot],
    model_params=model_params,
    name="Boltzmann Wealth Model",
)

# This is required to render the visualization in the Jupyter notebook
page

### Boltzmann Model Final Comments
## Simpler Simulation Models
### N-gram models

In [ ]:
#nltk.download('genesis')

kjv = nltk.corpus.genesis.words('english-kjv.txt')
bg = nltk.bigrams(kjv)
cfd = nltk.ConditionalFreqDist(bg)

In [ ]:
cfd['she']

In [ ]:
pp = pprint.PrettyPrinter(indent=4, compact=True,)
pp.pprint(' '.join(kjv[:20]))

In [ ]:
pp.pprint(cfd['And'].keys())

In [ ]:
rng = default_rng(1361)

def generate_model(cfdist, word, num=15):
    for i in range(num):
        print(word, end=' ')
        
        choices = np.array(list(cfdist[word].keys()))
        pp = np.array(list(cfdist[word].values()))
        pp = pp / np.sum(pp)
        word = rng.choice(choices, size=1, p = pp)[0]
        
generate_model(cfd, 'God', 10)

### Power Analysis
### Example: Power analysis with simulation

In [ ]:
rng_power = default_rng(4545)
def generate_one_sample(alpha, delta_m, sd1, n):
    X = rng_power.standard_normal(n)*sd1
    Y = rng_power.standard_normal(n)*sd1 + delta_m
    
    ts2 = stats.ttest_ind(X, Y)
    if ts2.pvalue < alpha:
        return 1 # 1 means reject H0
    else:
        return 0

In [ ]:
#| fig-align: center
#| label: fig-power
#| fig-cap: "Power analysis using simulation"
#| fig-pos: 'ht'
#| 
n_vals = np.arange(5, 50, step=4)

power_est = []
for n_ in n_vals:
    x = [generate_one_sample(0.05, 1, 1.2, n_) for ii in np.arange(0, 2000)]
    power_est.append(np.mean(x))
    # print("Done with sample size " + str(n_))
    
ax = plt.plot(n_vals, power_est, 'go-')
plt.hlines(0.9, n_vals[0], n_vals[-1], colors='b', linestyles='dotted')
plt.title('Power Estimates');

## Summary: Simulation-Based Modeling
## References
### Mesa Links
### Other ABM Software
### Reference Papers and Websites
### Other Simulation Software
### Other Links
## Exercises